# 05 — Discrimination, stabilité et calibration (étape 4)
Prédictions individuelles du walk-forward (cible principale `defaut_hors_forbearance`), recalculées une fois puis mises en cache dans `data/processed/preds_{AN}_principal.pkl`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE / "src"))
from validation_toolkit.credit.models import walk_forward
from validation_toolkit.core.discrimination import gini, ks, psi
from validation_toolkit.core.calibration_tests import binomial_exact, vasicek
from validation_toolkit.core.stat_tests import moyenne_newey_west

PROC = RACINE / "data" / "processed"

In [2]:
AN = 2015                              # 2007 ou 2015
PREMIERE_ANNEE_TEST = {2007: 2011, 2015: 2019}[AN]
CIBLE = "defaut_hors_forbearance"
SEUIL = 30                             # défauts minimum pour juger une année
RHOS = (0.05, 0.15, 0.24)              # corrélation d'actifs testée ; 0,24 = plafond cité par Tasche (2003) ; 0,15 à confirmer dans la réglementation

In [3]:
cache = PROC / f"preds_{AN}_principal.pkl"
if cache.exists():
    preds = pd.read_pickle(cache)
else:
    panel = pd.read_pickle(PROC / f"panel_{AN}.pkl")
    preds = walk_forward(panel, PREMIERE_ANNEE_TEST, cible=CIBLE)
    preds.to_pickle(cache)
preds.shape

(156219, 6)

## 1. Discrimination par année de test
`fiable` = au moins `SEUIL` défauts dans l'année ; sinon le Gini est du bruit.

In [4]:
lignes = []
for a, g in preds.groupby("annee"):
    r = {"annee": a, "n": len(g), "defauts": int(g[CIBLE].sum())}
    r["fiable"] = r["defauts"] >= SEUIL
    for m in ("ref", "chall"):
        ok = g[CIBLE].nunique() > 1
        r[f"gini_{m}"] = gini(g[CIBLE], g[f"pd_{m}"]) if ok else np.nan
        r[f"ks_{m}"] = ks(g[CIBLE], g[f"pd_{m}"]) if ok else np.nan
    lignes.append(r)
discri = pd.DataFrame(lignes)
display(discri.round(3))
print("Gini moyen sur les années fiables :", discri.query("fiable")[["gini_ref", "gini_chall"]].mean().round(3).to_dict())

,annee,n,defauts,fiable,gini_ref,ks_ref,gini_chall,ks_chall
0,2019,36337,103,True,0.606,0.465,0.540,0.460
1,2020,31943,62,True,0.464,0.375,0.428,0.336
2,2021,22450,20,False,0.549,0.452,0.517,0.429
3,2022,15951,29,False,0.416,0.393,0.386,0.370
4,2023,13904,29,False,0.415,0.338,0.320,0.327
5,2024,12876,30,True,0.370,0.390,0.277,0.280
6,2025,11883,29,False,0.241,0.279,0.239,0.280
7,2026,10875,16,False,0.236,0.281,0.440,0.386


Gini moyen sur les années fiables : {'gini_ref': 0.48, 'gini_chall': 0.415}


## 2. Stabilité (PSI)
Mesuré **à l'octroi** entre les deux millésimes (2007 = référence), sur les variables d'entrée. Mesurer le PSI d'un score d'une année de test à l'autre sur un seul millésime n'a pas de sens : il ne reste que les prêts survivants, donc la distribution change mécaniquement.
Repères usuels (conventions de métier, non statistiques) : < 0,10 stable ; 0,10–0,25 vigilance ; > 0,25 dérive.

In [5]:
def octroi(an):
    p = pd.read_pickle(PROC / f"panel_{an}.pkl")
    return p.drop_duplicates("id")

o07, o15 = octroi(2007), octroi(2015)
rows = []
for v in ("fico", "cltv", "ltv", "dti", "taux", "upb"):
    a, b = o07[v].dropna(), o15[v].dropna()
    rows.append({"variable": v, "moy_2007": a.mean(), "moy_2015": b.mean(), "psi": psi(a, b)})
display(pd.DataFrame(rows).round(3))

,variable,moy_2007,moy_2015,psi
0,fico,725.313,749.731,0.229
1,cltv,73.959,74.425,0.035
2,ltv,71.371,73.533,0.024
3,dti,36.616,33.834,1.318
4,taux,6.366,3.974,13.177
5,upb,182406.828,223607.052,0.121


## 3. Calibration globale par année : binomial contre Vasicek
Taux observé de l'année contre PD moyenne prédite, tous prêts. Deux sens : `sous` (le modèle sous-estime : risque pour la banque) et `sur` (le modèle sur-estime : prudence excessive). p < 0,05 = rejet de H0 « modèle bien calibré ».

In [6]:
def calibration_annuelle(preds, modele, cote):
    out = []
    for a, g in preds.groupby("annee"):
        n, d, pr = len(g), int(g[CIBLE].sum()), g[f"pd_{modele}"].mean()
        r = {"annee": a, "defauts": d, "obs": d / n, "pred": pr, "p_binom": binomial_exact(n, d, pr, cote)}
        for rho in RHOS:
            r[f"p_vas_{rho}"] = vasicek(n, d, pr, rho, cote)
        out.append(r)
    return pd.DataFrame(out)

cal = {(m, c): calibration_annuelle(preds, m, c) for m in ("ref", "chall") for c in ("sous", "sur")}
for (m, c), t in cal.items():
    pcols = [x for x in t.columns if x.startswith("p_")]
    print(f"\n--- modèle {m}, H1 = {c}-estimation — nombre d'années rejetées à 5 % :", (t[pcols] < 0.05).sum().to_dict())
display(cal[("chall", "sous")].round(4))
display(cal[("ref", "sur")].round(4))


--- modèle ref, H1 = sous-estimation — nombre d'années rejetées à 5 % : {'p_binom': 0, 'p_vas_0.05': 0, 'p_vas_0.15': 0, 'p_vas_0.24': 0}

--- modèle ref, H1 = sur-estimation — nombre d'années rejetées à 5 % : {'p_binom': 3, 'p_vas_0.05': 0, 'p_vas_0.15': 0, 'p_vas_0.24': 0}

--- modèle chall, H1 = sous-estimation — nombre d'années rejetées à 5 % : {'p_binom': 1, 'p_vas_0.05': 0, 'p_vas_0.15': 0, 'p_vas_0.24': 0}

--- modèle chall, H1 = sur-estimation — nombre d'années rejetées à 5 % : {'p_binom': 2, 'p_vas_0.05': 0, 'p_vas_0.15': 0, 'p_vas_0.24': 0}


,annee,defauts,obs,pred,p_binom,p_vas_0.05,p_vas_0.15,p_vas_0.24
0,2019,103,0.0028,0.0030,0.7226,0.4076,0.3045,0.2458
1,2020,62,0.0019,0.0035,1.0000,0.6947,0.4605,0.3542
2,2021,20,0.0009,0.0020,1.0000,0.7772,0.5060,0.3790
3,2022,29,0.0018,0.0009,0.0007,0.1068,0.1326,0.1193
4,2023,29,0.0021,0.0024,0.8008,0.4514,0.3225,0.2554
5,2024,30,0.0023,0.0027,0.8103,0.4560,0.3267,0.2595
6,2025,29,0.0024,0.0025,0.5732,0.3874,0.2917,0.2352
7,2026,16,0.0015,0.0014,0.4202,0.3316,0.2558,0.2051


,annee,defauts,obs,pred,p_binom,p_vas_0.05,p_vas_0.15,p_vas_0.24
0,2019,103,0.0028,0.0036,0.0052,0.4791,0.6355,0.7113
1,2020,62,0.0019,0.0036,0.0000,0.2914,0.5303,0.6390
2,2021,20,0.0009,0.0028,0.0000,0.1117,0.3907,0.5400
3,2022,29,0.0018,0.0020,0.3223,0.5715,0.6914,0.7559
4,2023,29,0.0021,0.0018,0.7890,0.6931,0.7522,0.7973
5,2024,30,0.0023,0.0018,0.9175,0.7450,0.7780,0.8146
6,2025,29,0.0024,0.0019,0.9218,0.7492,0.7797,0.8154
7,2026,16,0.0015,0.0020,0.1308,0.4661,0.6390,0.7210


## 4. Sensibilité de Newey-West au nombre de retards
Biais annuel moyen (observé − prédit) du modèle, test H0 : biais = 0.

In [7]:
serie = {m: preds.groupby("annee").apply(lambda g: g[CIBLE].mean() - g[f"pd_{m}"].mean(), include_groups=False)
         for m in ("ref", "chall")}
lignes = []
for m, s in serie.items():
    for L in (0, 1, 2, 3, 4):
        r = moyenne_newey_west(s.values, retards=L)
        lignes.append({"modele": m, "retards": L, "biais": r["moyenne"], "se": r["se"], "p": r["p"]})
display(pd.DataFrame(lignes).round(4))

,modele,retards,biais,se,p
0,ref,0,-0.0005,0.0003,0.1697
1,ref,1,-0.0005,0.0004,0.2620
2,ref,2,-0.0005,0.0004,0.2882
3,ref,3,-0.0005,0.0004,0.2790
4,ref,4,-0.0005,0.0004,0.2474
5,chall,0,-0.0003,0.0002,0.2335
6,chall,1,-0.0003,0.0002,0.2280
7,chall,2,-0.0003,0.0002,0.1571
8,chall,3,-0.0003,0.0002,0.1363
9,chall,4,-0.0003,0.0002,0.1337


## Limites à retenir
- Le Gini n'est pas interprétable sur les années à moins de 30 défauts.
- Le test de Vasicek dépend de ρ, qui est une hypothèse, pas une mesure ; les résultats sont donnés pour trois valeurs. Il ne traite pas la corrélation entre années.
- Le test de Vasicek est une approximation de grand portefeuille.
- Le PSI à l'octroi compare deux millésimes volontairement différents (avant / après crise) : un PSI élevé est attendu, pas une anomalie.
- Quand les supports de deux populations ne se recouvrent pas (par exemple `dti` jamais supérieur à 50 en 2015, ou les taux d'intérêt de 2007 et 2015), certaines classes sont vides et la valeur exacte du PSI dépend de l'epsilon qui évite le log(0). Seul l'ordre de grandeur (dérive forte) est à retenir ; sur `dti`, 7 classes fixes donnent 0,39 contre 1,32 avec les déciles de 2007.